# Chapter 11 — Context Management in Production Systems · Homework (self-test)

**Goal:** check that you can budget a context window, pick and use the right compaction mode, trim history without breaking it, keep transactional facts outside the history, and recover a long-running workflow after a crash, without looking at the answers.

- Theory: [README.md](README.md) · Practice: [01_practice.ipynb](01_practice.ipynb)
- **Estimated time:** 90–120 minutes
- **Estimated API cost:** $0 (every graded part runs offline)
- **Prerequisites:** the repo venv from [00-prerequisites](../../00-prerequisites/README.md). No API key is needed.

### How to use this notebook

1. Answer the **quiz** (Part A) in the `answers` cell, then run the grader. Answers are stored as hashes, so you cannot peek.
2. Solve each **coding exercise** (Part B): replace the `TODO` stub, then run its check cell. A check never crashes the notebook; it prints ✓ or a hint.
3. Write your answer to the **architecture scenario** (Part C) and compare it with the rubric.
4. Run the **scorecard**. The pass mark is **72%** (quiz questions + exercises combined), the same bar as the Claude path.
5. Solutions are at the very bottom. Try first.

In [ ]:
import copy
import hashlib
import json
import os
import tempfile
from pathlib import Path

from dotenv import find_dotenv, load_dotenv

load_dotenv(find_dotenv())
MODEL = os.getenv("OPENAI_MODEL", "gpt-6-luna")
HW_DIR = Path(tempfile.mkdtemp(prefix="oa_ch11_hw_"))  # scratch files stay outside the repo
EX_RESULTS: dict[int, bool] = {}
QUIZ_SCORE: dict[str, int] = {"correct": 0, "total": 10}


def record(n: int, passed: bool, hint: str = "") -> None:
    EX_RESULTS[n] = passed
    print(f"✓ Exercise {n} passed" if passed else f"✗ Exercise {n} not passed yet. {hint}")


def render_facts(facts: dict) -> str:
    """Given helper (same idea as the practice notebook): render a facts dict as a tagged block."""
    lines = [f"{k}: {v}" for k, v in facts.items() if v not in (None, [], "") and k != "version"]
    return f'<case_facts version="{facts.get("version", 1)}">\n' + "\n".join(lines) + "\n</case_facts>"


print("Model (not used by graded parts):", MODEL)

## Part A — Quiz (10 questions)

**Q1.** A research assistant on `gpt-6-astra` sends prompts of about 300,000 input tokens. Finance reports that each request costs roughly twice what the team estimated from the per-token prices. What is the most likely cause, and the best fix?

- A) Cache writes are billed at 1.25x the input rate; disable prompt caching.
- B) Above 272K input tokens the **whole** request is billed at 2x input and cache rates and 1.5x output; set a context budget below that threshold and enforce it with trimming, compaction or delegation.
- C) Reasoning tokens are billed on top of `output_tokens`; set the reasoning effort to the minimum.
- D) `previous_response_id` bills each turn twice; switch to the Conversations API.

**Q2.** A claims chatbot started returning HTTP 400 errors in very long conversations. A developer proposes setting `truncation="auto"` to make the errors go away. What should the architect say?

- A) Accept: `"auto"` keeps the first and the last items and drops only the middle of the conversation.
- B) Accept, and raise `max_output_tokens` so the model has room to remember.
- C) Retry the 400 with exponential backoff; it is a transient capacity error.
- D) Keep the default `"disabled"`; the 400 is a budget-policy bug. Measure with `input_tokens.count`, compact at a soft limit, and keep the transactional facts in a case-facts block.

**Q3.** A support conversation uses `previous_response_id` chaining. After 40 turns, a privacy request requires removing a message in which the customer typed their passport number, and the conversation must continue. What works?

- A) Start a new chain (or a stateless request) seeded with curated context (the case-facts block and the turns you keep, without that message), and delete the old stored responses according to your retention policy.
- B) Call `client.conversations.items.delete(...)` with the ID of the response that contains the message.
- C) Set `truncation="auto"`; the message will eventually drop out of the context.
- D) Send a developer message telling the model to ignore the passport number.

**Q4.** After calling `client.responses.compact(...)`, an engineer keeps only the compaction item from `compacted.output` and drops everything else in the returned window to save tokens. Is that correct?

- A) Yes: only the compaction item carries state.
- B) Yes, as long as the requests use `store=False`.
- C) No: the returned window is the canonical next context window. Pass it on as-is and append the new user message.
- D) No: you must also re-append the full original history after the compaction item.

**Q5.** An Agents SDK investigation agent receives one user request and then calls `fetch_logs` ten times **in the same run**; each result is about 10,000 characters, and the agent loses focus before it finishes. Which change actually shrinks what the model sees during that run?

- A) A `session_input_callback` that keeps only the last 10 stored items.
- B) `RunConfig(call_model_input_filter=ToolOutputTrimmer(recent_turns=1, trimmable_tools={"fetch_logs"}))`; it previews every output except the latest.
- C) Make `fetch_logs` return a bounded digest (counts, the matching error lines, a reference to the full log kept in your storage) so each result is small when it enters the context; if the run still grows, delegate the log analysis to an agent-as-tool that returns a structured summary, or rely on server-side compaction. Leave the items the model already received in this turn untouched.
- D) `ModelSettings(truncation="auto")` on the agent.

**Q6.** A stateless support assistant has a 150K-token history that is mostly served from the prompt cache. The case facts (order, amounts, status, commitments) change on almost every turn. Where should the case-facts block go?

- A) At the top of `instructions`, so it has the highest priority.
- B) Nowhere: ask the model to restate the key facts at the start of every answer.
- C) As a developer message appended to the stored history every turn.
- D) As an ephemeral developer message at the end of `input`, just before the new user message, rendered fresh each request and not saved into the stored history.

**Q7.** A team wraps its `SQLiteSession` with `OpenAIResponsesCompactionSession(session_id=..., underlying_session=store)` and no other arguments. In production, chats are compacted after almost every turn and the compaction calls show model `gpt-4.1`. What is the fix?

- A) Set `model=` explicitly and pass a `should_trigger_compaction` hook that reflects your own budget, because the defaults are `"gpt-4.1"` and compaction at 10 or more candidate items.
- B) Wrap an `OpenAIConversationsSession` instead, so OpenAI decides when to compact.
- C) Set `SessionSettings(limit=10)` so there are never enough items to compact.
- D) Switch `compaction_mode` to `"previous_response_id"`, which compacts less often.

**Q8.** A coding agent runs with automatic server-side compaction (`context_management` with `compact_threshold`). For one hard step, the team wants higher reasoning effort without rewriting the cached prefix, so a developer appends a `configuration_update` item. What is correct?

- A) Fine: `configuration_update` was designed to be used together with automatic compaction.
- B) Change `reasoning.effort` on the request instead; it does not affect the cache.
- C) Do not combine `configuration_update` with automatic compaction or truncation. Compact explicitly instead (a `compaction_trigger` item as the final input item), then add a fresh `configuration_update` before the next user message.
- D) Write "think harder" into `instructions`; it has the same effect as raising effort.

**Q9.** An architect designs a parallel code audit with **Responses Multi-agent** (beta) on `gpt-6.1-sol` and plans to call `/responses/compact` between audit phases to keep each subagent small. What should they know?

- A) Multi-agent is available on every GPT-6 model, including `gpt-6-luna`, and `/responses/compact` works per subagent.
- B) Subagents share one context window with the root agent, so compaction is applied once for the whole tree.
- C) Multi-agent disables compaction, so subagents must be kept short by prompt design.
- D) `/responses/compact` is not supported with Multi-agent; automatic server-side compaction is enabled implicitly and applied to the root and each subagent separately (`compact_threshold` can still be set).

**Q10.** A refund agent uses `SQLiteSession("case_1")` (no database path) and keeps paused runs in a Python dict. After a deploy, every pending approval is lost, and one refund was issued twice when a worker retried. Which design fixes both problems?

- A) Store the paused run's `previous_response_id` and resume from it after the deploy.
- B) A file-backed or shared session backend; persist `result.to_state().to_string()` plus a manifest; restore with `await RunState.from_string(agent, saved)`; give every refund an idempotency key and check its status in the payment system before retrying.
- C) Raise the worker shutdown timeout so deploys wait until every run finishes.
- D) On restart, re-run each workflow from the first user message; the model will notice that the refund was already issued.

In [ ]:
# Fill in "A", "B", "C" or "D" for each question.
answers = {
    "q1": None, "q2": None, "q3": None, "q4": None, "q5": None,
    "q6": None, "q7": None, "q8": None, "q9": None, "q10": None,
}

In [ ]:
ANSWER_KEY = {
    "q1": "96e2e2c94d02b534d4bdc303373df7b704ed4725981fac17549ee985d132c365",
    "q2": "d14b6ae0cbc51f4b715924684653e0d81035a8fe8b79de34ff29eb9db56e2d6b",
    "q3": "2501e2fe43d858f1773d414ee14ab0a8cdf2afefbc4ffd20ae5d956e618f3ea3",
    "q4": "69a9828043db3a177204403c987facd12c855928421b3aeeb5edce8407e3196c",
    "q5": "e3c6d5ddd445920945724cd165c914deaf97d5041f947ba2df35e569269b353f",
    "q6": "70425e0bfb4a573a030df7909d2bae441e3fdb1e7a565e097f3118231f4f27a9",
    "q7": "8722338db4eb12eaae5f6f26c68875af2b115581d770f01f98b9e0e1e6010ac1",
    "q8": "89443923f22dc808ba069cfb289117719cbe7d8d36703c74314c6b70fba0371b",
    "q9": "e58517837d895fe6365a5801be9a66d9d1d39f1d863920a84cbed549cfec1954",
    "q10": "c69a9867de9de4cc2e45f4cd6a6128833af714d8993edbeac622795c9c5e20d3",
}


def grade_quiz(answers: dict) -> int:
    correct = 0
    for qid in sorted(ANSWER_KEY, key=lambda q: int(q[1:])):
        given = str(answers.get(qid) or "").strip().upper()
        ok = hashlib.sha256(f"{qid}:{given}".encode()).hexdigest() == ANSWER_KEY[qid]
        correct += ok
        print(f"{qid:>4}: {'✓' if ok else '✗'}  (your answer: {given or '-'})")
    QUIZ_SCORE["correct"] = correct
    print(f"\nQuiz: {correct}/{len(ANSWER_KEY)} = {correct / len(ANSWER_KEY):.0%}")
    return correct


quiz_correct = grade_quiz(answers)

## Part B — Coding exercises (offline)

### Exercise 1 — A context budget policy

Write `next_action(input_tokens, *, window_ceiling=272_000, output_reserve=25_000, soft_ratio=0.6)` that returns one of `"send"`, `"compact"` or `"compact_now"`:

- `hard = window_ceiling - output_reserve` and `soft = soft_ratio * (window_ceiling - output_reserve)`
- `input_tokens >= hard` → `"compact_now"`; otherwise `input_tokens >= soft` → `"compact"`; otherwise `"send"`
- The defaults use the 272K **pricing** threshold, not the 1.05M technical window.

In [ ]:
def next_action(input_tokens: int, *, window_ceiling: int = 272_000, output_reserve: int = 25_000,
                soft_ratio: float = 0.6) -> str:
    # TODO: return "send", "compact" or "compact_now"
    raise NotImplementedError

In [ ]:
def check_ex1() -> None:
    try:
        cases = [(10_000, "send"), (148_199, "send"), (148_200, "compact"), (200_000, "compact"),
                 (246_999, "compact"), (247_000, "compact_now"), (300_000, "compact_now")]
        for tokens, want in cases:
            got = next_action(tokens)
            if got != want:
                hint = f"next_action({tokens:,}) returned {got!r}, expected {want!r}."
                if tokens == 148_200:
                    hint += (" With the defaults the soft limit is 0.6 x (272,000 - 25,000) = 148,200 and the check is >=."
                             " The ceiling is the 272K pricing threshold, not the 1.05M technical window.")
                if tokens == 247_000:
                    hint += " The hard limit is 272,000 - 25,000 = 247,000 and the check is >=. Did you subtract output_reserve?"
                raise AssertionError(hint)
        assert next_action(50_000, window_ceiling=100_000, output_reserve=10_000, soft_ratio=0.5) == "compact", \
            "Use the keyword arguments: with ceiling 100K, reserve 10K, ratio 0.5 the soft limit is 45K."
        assert next_action(90_000, window_ceiling=100_000, output_reserve=10_000) == "compact_now", \
            "Hard limit = window_ceiling - output_reserve (90K here)."
        record(1, True)
    except NotImplementedError:
        record(1, False, "Implement next_action first.")
    except Exception as e:  # never crash the notebook
        record(1, False, f"{type(e).__name__}: {e}")


check_ex1()

### Exercise 2 — A server-side compaction request builder

Write `build_compaction_request(model, instructions, history, user_message, threshold, previous_response_id=None)` that returns the keyword arguments for `client.responses.create(...)`:

- Always: `model`, `instructions` (it is **not** carried by `previous_response_id`) and `context_management=[{"type": "compaction", "compact_threshold": threshold}]` (a **list**).
- **Stateless** (`previous_response_id is None`): `input` = the whole `history` followed by `{"role": "user", "content": user_message}`, and `store=False`. No `previous_response_id` key.
- **Chained** (`previous_response_id` given): `previous_response_id`, and `input` = **only** the new user message. Do not set `store=False` (the next turn must be able to chain to this response) and do not replay the history.
- Never set `truncation="auto"`. Do not mutate `history`.

In [ ]:
def build_compaction_request(model: str, instructions: str, history: list[dict], user_message: str,
                             threshold: int, previous_response_id: str | None = None) -> dict:
    # TODO: build the responses.create kwargs
    raise NotImplementedError

In [ ]:
def check_ex2() -> None:
    try:
        hist = [{"role": "user", "content": "hi"}, {"role": "assistant", "content": "hello"}]
        snapshot = copy.deepcopy(hist)
        cm = [{"type": "compaction", "compact_threshold": 150_000}]
        s = build_compaction_request("gpt-6-luna", "RULES", hist, "next?", 150_000)
        assert hist == snapshot, "Do not mutate history; build a new list."
        assert s.get("context_management") == cm, f"context_management must be {cm} (a list), got {s.get('context_management')!r}"
        assert s.get("model") == "gpt-6-luna" and s.get("instructions") == "RULES", "Include model and instructions."
        assert s.get("store") is False, ("Stateless mode needs store=False: OpenAI keeps no Responses application state "
                                         "(abuse-monitoring logs still apply unless your org/project is approved for ZDR).")
        assert "previous_response_id" not in s, "Stateless requests must not contain previous_response_id."
        assert s.get("input", [])[:2] == hist and len(s["input"]) == 3, "Stateless input = full history + the new user message."
        last = s["input"][-1]
        assert last.get("role") == "user" and last.get("content") == "next?", "The new user message goes last."
        assert s.get("truncation") in (None, "disabled"), "Never set truncation='auto'."
        c = build_compaction_request("gpt-6-luna", "RULES", hist, "next?", 150_000, previous_response_id="resp_9")
        assert hist == snapshot, "Do not mutate history in chained mode either."
        assert c.get("model") == "gpt-6-luna", "Chained mode needs the model too: every request names its model."
        assert c.get("previous_response_id") == "resp_9", "Chained mode needs previous_response_id."
        assert c.get("input") == [{"role": "user", "content": "next?"}], \
            "Chained mode: input is ONLY the new user message, {'role': 'user', 'content': ...} (the server already holds the history; do not prune or replay)."
        assert c.get("store") is not False, "Chained mode: store=False would make this response impossible to chain to."
        assert c.get("instructions") == "RULES", "instructions are per request: send them on chained turns too."
        assert c.get("context_management") == cm, "Chained mode also needs context_management."
        assert c.get("truncation") in (None, "disabled"), "Never set truncation='auto'."
        record(2, True)
    except NotImplementedError:
        record(2, False, "Implement build_compaction_request first.")
    except Exception as e:
        record(2, False, f"{type(e).__name__}: {e}")


check_ex2()

### Exercise 3 — A field-level tool-output trimmer

Write `trim_tool_outputs(items, keep_fields, recent_turns=1, max_chars=300, preview_chars=80)` for a hand-written stateless loop. It returns a **new** list with the same items in the same order (never delete an item, never mutate the input):

- Only `function_call_output` items **before** the boundary are trimmed. The boundary is the index of the `recent_turns`-th user message counted from the end (`role == "user"`); if there are fewer user messages, nothing is trimmed.
- The tool name of an output comes from the `function_call` item with the same `call_id`.
- If the output parses as a JSON object that has an `"error"` key, keep it unchanged (errors stay readable).
- Else, if it parses as a JSON object and the tool name is in `keep_fields`, replace it with `json.dumps` of only those fields (field-level trim).
- Else, if the output string is longer than `max_chars`, replace it with a preview that starts with `"[Trimmed"` and contains the first `preview_chars` characters of the original.
- Otherwise leave it unchanged.

In [ ]:
def trim_tool_outputs(items: list[dict], keep_fields: dict[str, list[str]], recent_turns: int = 1,
                      max_chars: int = 300, preview_chars: int = 80) -> list[dict]:
    # TODO: trim older function_call_output items, keep pairs and the current turn intact
    raise NotImplementedError

In [ ]:
def _ex3_history() -> list[dict]:
    order = {"order_id": "A-1", "status": "shipped", "total": "74.95 USD", **{f"f{i}": "x" * 20 for i in range(30)}}
    return [
        {"role": "user", "content": "Where is A-1?"},
        {"type": "function_call", "call_id": "c1", "name": "lookup_order", "arguments": '{"order_id": "A-1"}'},
        {"type": "function_call_output", "call_id": "c1", "output": json.dumps(order)},
        {"role": "assistant", "content": "Shipped."},
        {"role": "user", "content": "And B-2? Also search the return policy."},
        {"type": "function_call", "call_id": "c2", "name": "lookup_order", "arguments": '{"order_id": "B-2"}'},
        {"type": "function_call_output", "call_id": "c2", "output": json.dumps({"error": "not found", "retryable": False, "detail": "y" * 400})},
        {"type": "function_call", "call_id": "c3", "name": "search_kb", "arguments": '{"query": "returns"}'},
        {"type": "function_call_output", "call_id": "c3", "output": "Return policy: " + "30 days. " * 80},
        {"type": "function_call", "call_id": "c4", "name": "search_kb", "arguments": '{"query": "short"}'},
        {"type": "function_call_output", "call_id": "c4", "output": "short answer"},
        {"role": "assistant", "content": "B-2 not found; returns within 30 days."},
        {"role": "user", "content": "Thanks, and A-1 again?"},
        {"type": "function_call", "call_id": "c5", "name": "lookup_order", "arguments": '{"order_id": "A-1"}'},
        {"type": "function_call_output", "call_id": "c5", "output": json.dumps(order)},
    ]


def check_ex3() -> None:
    try:
        hist = _ex3_history()
        snapshot = copy.deepcopy(hist)
        keep = {"lookup_order": ["order_id", "status", "total"]}
        out = trim_tool_outputs(hist, keep, recent_turns=1, max_chars=300, preview_chars=80)
        assert hist == snapshot, "Do not mutate the input items; build new dicts for trimmed outputs."
        assert isinstance(out, list) and len(out) == len(hist), "Never delete items: the result must have the same length."
        assert [i.get("call_id") for i in out] == [i.get("call_id") for i in hist], "Keep every item (and call_id) in its place."
        for i, (o, h) in enumerate(zip(out, hist)):
            if h.get("type") != "function_call_output":
                assert o == h, f"Item {i} ({h.get('type') or h.get('role')}) changed: only function_call_output 'output' strings may be trimmed."
            else:
                assert {k: v for k, v in o.items() if k != "output"} == {k: v for k, v in h.items() if k != "output"}, \
                    f"Item {i}: only the 'output' field of a function_call_output may change (keep type and call_id)."
        assert json.loads(out[2]["output"]) == {"order_id": "A-1", "status": "shipped", "total": "74.95 USD"}, \
            "Old lookup_order output: keep only the fields listed in keep_fields for that tool."
        assert out[6] == hist[6], "Error outputs must stay unchanged and readable."
        assert out[8]["output"].startswith("[Trimmed") and hist[8]["output"][:80] in out[8]["output"] \
            and len(out[8]["output"]) < len(hist[8]["output"]), "Long non-JSON outputs become a '[Trimmed ...' preview with the first preview_chars characters."
        assert out[10] == hist[10], "Short outputs stay unchanged."
        assert out[14] == hist[14], "Do not touch the current turn (everything from the boundary user message on)."
        out2 = trim_tool_outputs(hist, keep, recent_turns=2)
        assert out2[2]["output"] != hist[2]["output"] and out2[8] == hist[8], \
            "With recent_turns=2 the boundary is the 2nd user message from the end: only the first turn is trimmed."
        assert trim_tool_outputs(hist[:3], keep, recent_turns=2) == hist[:3], "Fewer user messages than recent_turns: trim nothing."
        record(3, True)
    except NotImplementedError:
        record(3, False, "Implement trim_tool_outputs first.")
    except Exception as e:
        record(3, False, f"{type(e).__name__}: {e}")


check_ex3()

### Exercise 4 — Case-facts merge and position-aware input

Write two functions.

`merge_facts(facts, delta, source)` returns a **new** facts dict (do not mutate `facts`). `facts` always has `"version"` (int), `"commitments"` (list) and `"conflicts"` (list).

- Ignore keys whose value in `delta` is `None`.
- `"commitments"`: append each new entry that is not already present.
- `TOOL_OWNED = {"order_id", "order_total", "order_date"}`: if `source == "user"` and the current value is not `None` and differs, **do not overwrite**; append a conflict string that contains the field name to `"conflicts"`.
- Anything else: set the value.
- Increase `"version"` by 1 if and only if something changed (a value set, a commitment added or a conflict recorded).

`assemble_input(history, facts, user_message)` returns `[*history, {"role": "developer", "content": render_facts(facts)}, {"role": "user", "content": user_message}]` (stable history first, the ephemeral facts block and the question last). Do not mutate `history`, and do not put the facts anywhere else.

In [ ]:
TOOL_OWNED = {"order_id", "order_total", "order_date"}


def merge_facts(facts: dict, delta: dict, source: str) -> dict:
    # TODO: merge delta into a copy of facts, recording conflicts and bumping the version
    raise NotImplementedError


def assemble_input(history: list[dict], facts: dict, user_message: str) -> list[dict]:
    # TODO: history, then the facts block (developer), then the user message
    raise NotImplementedError

In [ ]:
def check_ex4() -> None:
    try:
        base = {"customer_id": "C-1", "order_id": None, "order_total": None, "issue": None,
                "commitments": [], "conflicts": [], "version": 1}
        snap = copy.deepcopy(base)
        f1 = merge_facts(base, {"order_id": "A-1", "order_total": "74.95 USD", "issue": None}, "tool")
        assert base == snap, "Do not mutate the facts you were given; return a new dict."
        assert f1["order_total"] == "74.95 USD" and f1["order_id"] == "A-1" and f1["issue"] is None, \
            "Tool values are set; None values in the delta are ignored."
        assert f1["version"] == 2, "One merge that changed something bumps the version by exactly 1."
        f2 = merge_facts(f1, {"order_total": "120 USD"}, "user")
        assert f2["order_total"] == "74.95 USD", "A user may not overwrite a tool-owned value that differs."
        assert any("order_total" in c for c in f2["conflicts"]), "Record the disagreement in 'conflicts' (mention the field)."
        assert f2["version"] == 3, "Recording a conflict is a change: bump the version."
        assert f1["conflicts"] == [], "merge_facts changed the previous facts' conflicts list: copy nested lists too (copy.deepcopy)."
        f3 = merge_facts(f2, {"issue": "torn tent", "commitments": ["refund 74.95 USD by 2026-10-10"]}, "user")
        assert f3["issue"] == "torn tent" and f3["commitments"] == ["refund 74.95 USD by 2026-10-10"], \
            "Users may set non-tool-owned fields; commitments are appended."
        f4 = merge_facts(f3, {"commitments": ["refund 74.95 USD by 2026-10-10"], "issue": "torn tent", "order_id": None}, "agent")
        assert f4["commitments"] == f3["commitments"] and f4["version"] == f3["version"], \
            "No real change (duplicate commitment, same value, None) means no version bump."
        f5 = merge_facts(f3, {"order_total": "90.00 USD"}, "tool")
        assert f5["order_total"] == "90.00 USD", "The system of record (source='tool') may correct a value."
        hist = [{"role": "user", "content": "hi"}, {"role": "assistant", "content": "hello"}]
        hsnap = copy.deepcopy(hist)
        req = assemble_input(hist, f3, "Can I get the refund today?")
        assert hist == hsnap, "Do not mutate history: the facts block is ephemeral."
        assert req[:2] == hist, "Stable history first (append-only, cache-friendly)."
        assert len(req) == 4, "Exactly: history + facts block + user message."
        assert req[2] == {"role": "developer", "content": render_facts(f3)}, "The second-to-last item is the facts block: a developer message rendered with render_facts."
        assert req[3] == {"role": "user", "content": "Can I get the refund today?"}, "The new user message goes last."
        record(4, True)
    except NotImplementedError:
        record(4, False, "Implement merge_facts and assemble_input first.")
    except Exception as e:
        record(4, False, f"{type(e).__name__}: {e}")


check_ex4()

### Exercise 5 — Pruning before the latest compaction item

Write `prune_for_next_request(items, mode)` that returns the item list to send next (a new list; do not mutate `items`):

- `mode == "stateless"`: drop every item **before the most recent** compaction item (`type == "compaction"`); keep that compaction item and everything after it. Without a compaction item, return the items unchanged.
- `mode == "previous_response_id"`: return the items unchanged (never prune a chain manually).
- `mode == "standalone"` (the window returned by `/responses/compact`): return the items unchanged (it is the canonical next window).
- Any other mode: raise `ValueError`.

In [ ]:
def prune_for_next_request(items: list[dict], mode: str) -> list[dict]:
    # TODO: apply the pruning rules for each compaction mode
    raise NotImplementedError

In [ ]:
def check_ex5() -> None:
    try:
        u = lambda t: {"role": "user", "content": t}
        c1 = {"type": "compaction", "id": "cmp_1", "encrypted_content": "e1"}
        c2 = {"type": "compaction", "id": "cmp_2", "encrypted_content": "e2"}
        items = [u("a"), {"role": "assistant", "content": "b"}, c1, u("c"), {"role": "assistant", "content": "d"}, c2,
                 {"role": "assistant", "content": "e"}, u("f")]
        snap = copy.deepcopy(items)
        got = prune_for_next_request(items, "stateless")
        assert items == snap, "Do not mutate items."
        assert got and got[0] == c2, "Keep the MOST RECENT compaction item as the first item (not the first one, and do not drop it)."
        assert got == items[5:], "Drop only the items before the latest compaction item; keep everything after it."
        plain = [u("a"), {"role": "assistant", "content": "b"}]
        assert prune_for_next_request(plain, "stateless") == plain, "No compaction item: nothing to prune."
        assert prune_for_next_request(items, "previous_response_id") == items, "Never prune manually with previous_response_id chaining."
        assert prune_for_next_request(items, "standalone") == items, "The output of /responses/compact is passed on as-is."
        try:
            prune_for_next_request(items, "magic")
            raise AssertionError("Unknown modes should raise ValueError.")
        except ValueError:
            pass
        record(5, True)
    except NotImplementedError:
        record(5, False, "Implement prune_for_next_request first.")
    except Exception as e:
        record(5, False, f"{type(e).__name__}: {e}")


check_ex5()

### Exercise 6 — Checkpoint and restore with an idempotent side effect

A refund workflow checkpoints its state as JSON:

```json
{"phases": {"triage": "completed", "refund": "in_progress", "reply": "not_started"},
 "side_effects": [{"id": "refund-4711-1", "status": "requested", "order_id": "A-1", "amount": "74.95 USD"}]}
```

Write two functions:

- `save_checkpoint(path, state)`: write `state` as JSON **atomically**: write a temporary file next to `path`, then move it over `path` with `os.replace` (or `Path.replace`). No temporary file may be left behind.
- `restore_and_resume(path, gateway)`: load the checkpoint and return a list of actions, in this order:
  1. For each side effect with status `"requested"` (in list order): if `gateway.status(id) == "completed"`, mark it `"completed"` **without** calling the gateway again and add `f"verified:{id}"`; otherwise call `gateway.refund(id, order_id, amount)`, mark it `"completed"` and add `f"issued:{id}"`. Side effects that are already `"completed"` are skipped.
  2. For each phase whose status is not `"completed"` (in dict order), add `f"resume:{phase}"`.
  3. Save the updated state with `save_checkpoint` before returning.

In [ ]:
def save_checkpoint(path: Path, state: dict) -> None:
    # TODO: atomic write (temp file + os.replace)
    raise NotImplementedError


def restore_and_resume(path: Path, gateway) -> list[str]:
    # TODO: verify or issue pending side effects, list phases to resume, save the updated state
    raise NotImplementedError

In [ ]:
class FakeGateway:
    def __init__(self, completed: set[str] = frozenset()):
        self.done = {k: "completed" for k in completed}
        self.refund_calls: list[str] = []

    def status(self, key: str) -> str | None:
        return self.done.get(key)

    def refund(self, key: str, order_id: str, amount: str) -> dict:
        self.refund_calls.append(key)
        self.done[key] = "completed"
        return {"id": key, "status": "completed"}


def check_ex6() -> None:
    try:
        import os as _os
        d = Path(tempfile.mkdtemp(dir=HW_DIR))
        path = d / "case.json"
        state = {"phases": {"triage": "completed", "refund": "in_progress", "reply": "not_started"},
                 "side_effects": [
                     {"id": "refund-1", "status": "requested", "order_id": "A-1", "amount": "74.95 USD"},
                     {"id": "refund-2", "status": "requested", "order_id": "A-2", "amount": "15.00 USD"},
                     {"id": "refund-0", "status": "completed", "order_id": "A-0", "amount": "5.00 USD"}]}
        moves = []
        real_replace, real_rename = _os.replace, _os.rename
        _os.replace = lambda s, t, *a, **k: (moves.append(str(t)), real_replace(s, t, *a, **k))[1]
        _os.rename = lambda s, t, *a, **k: (moves.append(str(t)), real_rename(s, t, *a, **k))[1]
        try:
            save_checkpoint(path, state)
        finally:
            _os.replace, _os.rename = real_replace, real_rename
        assert json.loads(path.read_text()) == state, "save_checkpoint must write the state as JSON to path."
        assert str(path) in moves, "Write a temp file and move it over the target with os.replace (atomic), not straight to path."
        assert [p.name for p in d.iterdir()] == ["case.json"], "No temporary file may be left behind."
        try:
            save_checkpoint(path, {"bad": object()})          # not JSON-serializable: the save fails...
        except Exception:
            pass
        assert json.loads(path.read_text()) == state, "...but a failed save must leave the previous checkpoint intact."
        for p in d.iterdir():
            if p.name != "case.json":
                p.unlink()                                       # tolerate a leftover temp file from the failed save
        gw = FakeGateway(completed={"refund-1"})               # refund-1 went through just before the crash
        actions = restore_and_resume(path, gw)
        assert gw.refund_calls == ["refund-2"], \
            f"Only refund-2 should be issued; gateway.refund was called for {gw.refund_calls}. Check the gateway status first; skip completed ones."
        assert actions == ["verified:refund-1", "issued:refund-2", "resume:refund", "resume:reply"], \
            f"Unexpected actions {actions}. Expected verified/issued in side-effect order, then resume:<phase> for unfinished phases."
        saved = json.loads(path.read_text())
        assert all(se["status"] == "completed" for se in saved["side_effects"]), "Save the updated statuses before returning."
        gw2 = FakeGateway()
        again = restore_and_resume(path, gw2)
        assert gw2.refund_calls == [] and again == ["resume:refund", "resume:reply"], \
            "Restoring twice must not repeat side effects (idempotent recovery)."
        record(6, True)
    except NotImplementedError:
        record(6, False, "Implement save_checkpoint and restore_and_resume first.")
    except Exception as e:
        record(6, False, f"{type(e).__name__}: {e}")


check_ex6()

## Part C — Architecture scenario

A European telecom wants a **technical-support assistant** built on the Responses API and the Agents SDK:

- Cases stay open for days or weeks; customers return through web chat and through phone transcripts that are added to the same case.
- EU customers require **Zero Data Retention**. The backend runs as several stateless workers behind a queue.
- The main tool, `run_diagnostics`, returns about 20,000 characters of modem and line logs per call, and the agent often calls it several times in one turn.
- Service credits above €50 need a supervisor's approval, which can take hours.
- Auditors must be able to see exactly which credits and appointments the assistant promised, and when.
- The budget target is under €0.05 per case-day on average.

Describe your design: where conversation state lives, how you keep the context within budget (measurement, trimming, compaction, delegation), where the transactional facts live and how they are placed in each request, how approvals and crashes are handled, and how you keep prompt caching effective. Name at least two trade-offs.

*Your answer:*

-
-
-

<details><summary>Rubric: what a strong answer contains</summary>

- **State strategy.** Zero Data Retention approved for the EU organization or project (`store=False` alone is not ZDR: it avoids stored Responses application state, but prompts and responses still go to abuse-monitoring logs for up to 30 days by default). Stateless requests with `store=False` (under ZDR `store` is treated as false anyway, and the Conversations API is not ZDR-eligible), with history in the telecom's own shared store (for example a SQL or Redis Agents SDK session, never in-memory `SQLiteSession`) so any worker can pick up a case. Replay every output item, including encrypted reasoning items. One strategy per case, no mixing with `previous_response_id`.
- **Budget.** Per-turn measurement with `responses.input_tokens.count` before sending and `usage` after; a soft limit (compact) and a hard limit (never send) below the 272K pricing threshold with an output reserve; `truncation` left at `"disabled"`; overflow 400s treated as bugs.
- **Trimming.** `run_diagnostics` returns only the fields the task needs (a structured digest, with the full logs in the telecom's own store behind a reference), at the source, so each result is small when it enters the context. Within one run, items already returned in the current turn stay untouched (reasoning guide); growth inside a turn is handled by the digest, delegation or server-side compaction. Across turns, `ToolOutputTrimmer` previews older outputs, wrapped by application code that keeps error outputs readable (the SDK trimmer has no error exemption). Call/output pairs stay intact.
- **Compaction.** Server-side `context_management` (ZDR-friendly with `store=False`) or standalone `responses.compact` at a phase boundary; in stateless mode the items before the latest compaction item may be dropped; the compacted window is never pruned. Mentions that the compaction item is opaque.
- **Case facts.** A versioned, application-owned facts record (customer, line ID, fault, appointments, credits, commitments with dates) updated from tool results and a small structured extraction, with conflicts recorded and logged per change: that is the audit trail. Rendered as an ephemeral developer message at the end of each request (stable prefix stays cacheable).
- **Delegation.** Optional: a diagnostics sub-agent (agent-as-tool with `custom_output_extractor`) returns a structured verdict so the coordinator's context stays small; cost trade-off acknowledged.
- **Approvals and crashes.** Credits above €50 via `needs_approval` → `result.interruptions` → `to_state().to_string()` persisted with a manifest → `await RunState.from_string(...)` in whichever worker picks the approval up; idempotency keys on credits, with status checks before retry; atomic checkpoints after each step; fresh session seeded with facts and "what changed" when diagnostics are stale.
- **Caching.** Stable instructions and tools first, append-only history between compactions, trimming batched with compaction, `configuration_update` (not a changed `reasoning.effort`) if effort must change, and measurement of `cached_tokens` / `cache_write_tokens` against the €0.05 target on `gpt-6-luna` pricing.
- **Trade-offs (at least two).** For example: compaction resets part of the cache but shrinks every later request; opaque compaction vs. your own readable summary; delegation saves the coordinator's context but adds total tokens; ZDR/stateless gives control but more payload per request and more engineering.

</details>

## Scorecard

In [ ]:
quiz_correct, quiz_total = QUIZ_SCORE["correct"], QUIZ_SCORE["total"]
ex_total = 6
ex_passed = sum(EX_RESULTS.get(n, False) for n in range(1, ex_total + 1))
overall = (quiz_correct + ex_passed) / (quiz_total + ex_total)
print(f"Quiz:      {quiz_correct}/{quiz_total}")
print(f"Exercises: {ex_passed}/{ex_total}  " + " ".join(f"Ex{n}:{'✓' if EX_RESULTS.get(n) else '✗'}" for n in range(1, ex_total + 1)))
print(f"Overall:   {overall:.0%}  ->  {'PASS' if overall >= 0.72 else 'RETRY'} (pass mark 72%)")
print("Part C is self-assessed against the rubric.")

Next chapter: Chapter 12 — Provenance: citations and annotations (coming soon, see the [planned chapter folders](../README.md#planned-chapter-folders)) · Back to the [OpenAI path](../README.md)

---

## Solutions (spoilers)

<details><summary>Quiz answers with explanations</summary>

- **Q1 — B.** On the GPT-6 models, prompts above 272K input tokens are billed at 2x input and cache rates and 1.5x output for the full request. A: cache writes are 1.25x, but that does not double the bill, and caching lowers the repeated-prefix price. C: reasoning tokens are already part of `output_tokens`. D: chaining re-bills the earlier input as input tokens, it does not bill twice.
- **Q2 — D.** `"auto"` drops items from the **beginning** of the conversation (A is wrong), exactly where account and order numbers usually are, and it does so silently. B changes output length only. C: an overflow is deterministic, not transient.
- **Q3 — A.** A `previous_response_id` chain is immutable; to drop content you start a new chain or a stateless request seeded with the curated context. B: Conversations items belong to Conversation objects, not to a response chain. C drops content blindly and only eventually. D leaves the data in the context.
- **Q4 — C.** The compaction guide says not to prune `/responses/compact` output: the returned window is the canonical next window and can include retained items. A and B prune it anyway. D re-adds what compaction just replaced.
- **Q5 — C.** Everything here happens inside one user turn, and OpenAI's reasoning guide says that when you truncate context, all items between the last user message and the latest function call output must be passed back untouched. So the growth has to be stopped **before** a result enters the context (a bounded digest at the source), or handled by delegation or supported compaction. B looks right but `ToolOutputTrimmer` never trims at or after the most recent `recent_turns` user messages (and `recent_turns` is at least 1), so in a single run it trims nothing, by design. A runs once per run, before the loop. D drops the oldest items blindly.
- **Q6 — D.** Facts near the question help attention, and keeping them out of the stored history keeps the long prefix byte-identical for caching. A breaks the cached prefix on every change. B stores facts inside the history that gets compacted. C accumulates stale copies.
- **Q7 — A.** The defaults are `model="gpt-4.1"` and a trigger at 10 or more candidate items (everything except user messages and compaction items). B raises `ValueError`: the wrapper cannot wrap `OpenAIConversationsSession`. C limits retrieval, not compaction policy. D changes how history is sent, not when compaction runs.
- **Q8 — C.** The reasoning guide says not to combine `configuration_update` with automatic compaction or truncation (and `/responses/compact` rejects such histories); compact explicitly with a `compaction_trigger` item, then add a fresh `configuration_update`. B rewrites the prefix (a cache miss). D is not an effort control.
- **Q9 — D.** The Multi-agent guide lists exactly these limitations: no `/responses/compact`, implicit automatic compaction per agent, `compact_threshold` still configurable. A: Multi-agent is on `gpt-6.1-sol` and the GPT-5.6 models, not Luna. B: each subagent keeps its own context. C is the opposite of the documented behavior.
- **Q10 — B.** Durable storage plus serialized run state survives deploys, and idempotency keys plus status checks prevent double refunds. A: a response ID is not the SDK's run state and does not prevent duplicates. C fails on crashes. D repeats side effects and trusts the model to prevent duplicates.

</details>

<details><summary>Exercise 1 — reference solution</summary>

```python
def next_action(input_tokens: int, *, window_ceiling: int = 272_000, output_reserve: int = 25_000,
                soft_ratio: float = 0.6) -> str:
    hard = window_ceiling - output_reserve
    soft = soft_ratio * (window_ceiling - output_reserve)
    if input_tokens >= hard:
        return "compact_now"
    if input_tokens >= soft:
        return "compact"
    return "send"
```

</details>

<details><summary>Exercise 2 — reference solution</summary>

```python
def build_compaction_request(model: str, instructions: str, history: list[dict], user_message: str,
                             threshold: int, previous_response_id: str | None = None) -> dict:
    user = {"role": "user", "content": user_message}
    request = {
        "model": model,
        "instructions": instructions,
        "context_management": [{"type": "compaction", "compact_threshold": threshold}],
    }
    if previous_response_id is None:
        request.update(input=[*history, user], store=False)
    else:
        request.update(previous_response_id=previous_response_id, input=[user])
    return request
```

</details>

<details><summary>Exercise 3 — reference solution</summary>

```python
def trim_tool_outputs(items: list[dict], keep_fields: dict[str, list[str]], recent_turns: int = 1,
                      max_chars: int = 300, preview_chars: int = 80) -> list[dict]:
    user_idx = [i for i, it in enumerate(items) if it.get("role") == "user"]
    boundary = user_idx[-recent_turns] if len(user_idx) >= recent_turns else 0
    names = {it["call_id"]: it["name"] for it in items if it.get("type") == "function_call"}
    out = []
    for i, it in enumerate(items):
        if i < boundary and it.get("type") == "function_call_output" and isinstance(it.get("output"), str):
            text, name = it["output"], names.get(it["call_id"], "")
            try:
                data = json.loads(text)
            except ValueError:
                data = None
            if isinstance(data, dict) and "error" in data:
                pass                                            # errors stay readable
            elif isinstance(data, dict) and name in keep_fields:
                it = {**it, "output": json.dumps({k: data[k] for k in keep_fields[name] if k in data})}
            elif len(text) > max_chars:
                it = {**it, "output": f"[Trimmed: {name} output — {len(text)} chars]\n{text[:preview_chars]}..."}
        out.append(it)
    return out
```

</details>

<details><summary>Exercise 4 — reference solution</summary>

```python
TOOL_OWNED = {"order_id", "order_total", "order_date"}


def merge_facts(facts: dict, delta: dict, source: str) -> dict:
    new = copy.deepcopy(facts)
    changed = False
    for key, value in delta.items():
        if value is None:
            continue
        if key == "commitments":
            for c in value:
                if c not in new["commitments"]:
                    new["commitments"].append(c)
                    changed = True
            continue
        current = new.get(key)
        if current == value:
            continue
        if source == "user" and key in TOOL_OWNED and current is not None:
            new["conflicts"].append(f"{key}: user says {value!r}, system of record says {current!r}")
            changed = True
            continue
        new[key] = value
        changed = True
    if changed:
        new["version"] += 1
    return new


def assemble_input(history: list[dict], facts: dict, user_message: str) -> list[dict]:
    return [*history,
            {"role": "developer", "content": render_facts(facts)},
            {"role": "user", "content": user_message}]
```

</details>

<details><summary>Exercise 5 — reference solution</summary>

```python
def prune_for_next_request(items: list[dict], mode: str) -> list[dict]:
    if mode in ("previous_response_id", "standalone"):
        return list(items)
    if mode != "stateless":
        raise ValueError(f"unknown mode: {mode}")
    last = max((i for i, it in enumerate(items) if it.get("type") == "compaction"), default=None)
    return list(items) if last is None else items[last:]
```

</details>

<details><summary>Exercise 6 — reference solution</summary>

```python
def save_checkpoint(path: Path, state: dict) -> None:
    path = Path(path)
    tmp = path.with_name(path.name + ".tmp")
    try:
        tmp.write_text(json.dumps(state, indent=2))
        os.replace(tmp, path)
    finally:
        if tmp.exists():
            tmp.unlink()


def restore_and_resume(path: Path, gateway) -> list[str]:
    state = json.loads(Path(path).read_text())
    actions = []
    for se in state["side_effects"]:
        if se["status"] != "requested":
            continue
        if gateway.status(se["id"]) == "completed":
            actions.append(f"verified:{se['id']}")
        else:
            gateway.refund(se["id"], se["order_id"], se["amount"])
            actions.append(f"issued:{se['id']}")
        se["status"] = "completed"
    actions += [f"resume:{phase}" for phase, status in state["phases"].items() if status != "completed"]
    save_checkpoint(path, state)
    return actions
```

</details>